# Tools и Agents

В ноутбуке собран путь от ручного форматирования prompt до tool calling и простого ReAct-агента.


In [ ]:
%pip install -qU "together>=1.1.0" transformers requests yfinance langchain langchain-community langchain-core langchain-together llama-index-core llama-index-llms-together


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from datetime import datetime, timedelta
from random import sample
from typing import Dict, List, Tuple

import requests
from together import Together
from transformers import AutoTokenizer

/Users/kirill/PycharmProjects/mentor_nizov/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3.1-70B")

## Ручное форматирование промпта

Здесь вручную собирается prompt в формате Llama 3.1 из списка сообщений с ролями.


In [4]:
def format_messages_to_prompt(messages: List[Dict[str, str]]) -> str:
    builder = ""
    builder += tokenizer.bos_token
    for message in messages:
        role = message["role"]
        content = message["content"]
        builder += f"<|start_header_id|>{role}<|end_header_id|>\n\n"
        builder += content + "<|eot_id|>"
    return builder


messages = [
    {"role": "system", "content": "Some system message"},
    {"role": "user", "content": "This is a message from the user"},
    {"role": "assistant", "content": "this is a mesage from the assistant"}
]

reference_text = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Some system message<|eot_id|><|start_header_id|>user<|end_header_id|>

This is a message from the user<|eot_id|><|start_header_id|>assistant<|end_header_id|>

this is a mesage from the assistant<|eot_id|>"""

assert format_messages_to_prompt(messages) == reference_text

Здесь в токенайзер добавляется chat template для Llama 3.1.


In [5]:
chat_template = """
{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- set date_string = "26 Jul 2024" %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message + builtin tools #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if builtin_tools is defined or tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{%- if builtin_tools is defined %}
    {{- "Tools: " + builtin_tools | reject('equalto', 'code_interpreter') | join(", ") + "\n\n"}}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- "Today Date: " + date_string + "\n\n" }}
{%- if tools is not none and not tools_in_user_message %}
    {{- "You have access to the following functions. To call a function, please respond with JSON for a function call." }}
    {{- 'Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}.' }}
    {{- "Do not use variables.\n\n" }}
    {%- for t in tools %}
        {{- t | tojson(indent=4) }}
        {{- "\n\n" }}
    {%- endfor %}
{%- endif %}
{{- system_message }}
{{- "<|eot_id|>" }}

{#- Custom tools are passed in a user message with some extra guidance #}
{%- if tools_in_user_message and not tools is none %}
    {#- Extract the first user message so we can plug it in here #}
    {%- if messages | length != 0 %}
        {%- set first_user_message = messages[0]['content']|trim %}
        {%- set messages = messages[1:] %}
    {%- else %}
        {{- raise_exception("Cannot put tools in the first user message when there's no first user message!") }}
{%- endif %}
    {{- '<|start_header_id|>user<|end_header_id|>\n\n' -}}
    {{- "Given the following functions, please respond with a JSON for a function call " }}
    {{- "with its proper arguments that best answers the given prompt.\n\n" }}
    {{- 'Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}.' }}
    {{- "Do not use variables.\n\n" }}
    {%- for t in tools %}
        {{- t | tojson(indent=4) }}
        {{- "\n\n" }}
    {%- endfor %}
    {{- first_user_message + "<|eot_id|>"}}
{%- endif %}

{%- for message in messages %}
    {%- if not (message.role == 'ipython' or message.role == 'tool' or 'tool_calls' in message) %}
        {{- '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' }}
    {%- elif 'tool_calls' in message %}
        {%- if not message.tool_calls|length == 1 %}
            {{- raise_exception("This model only supports single tool-calls at once!") }}
        {%- endif %}
        {%- set tool_call = message.tool_calls[0].function %}
        {%- if builtin_tools is defined and tool_call.name in builtin_tools %}
            {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' -}}
            {{- "<|python_tag|>" + tool_call.name + ".call(" }}
            {%- for arg_name, arg_val in tool_call.arguments | items %}
                {{- arg_name + '="' + arg_val + '"' }}
                {%- if not loop.last %}
                    {{- ", " }}
                {%- endif %}
                {%- endfor %}
            {{- ")" }}
        {%- else  %}
            {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' -}}
            {{- '{"name": "' + tool_call.name + '", ' }}
            {{- '"parameters": ' }}
            {{- tool_call.arguments | tojson }}
            {{- "}" }}
        {%- endif %}
        {%- if builtin_tools is defined %}
            {#- This means we're in ipython mode #}
            {{- "<|eom_id|>" }}
        {%- else %}
            {{- "<|eot_id|>" }}
        {%- endif %}
    {%- elif message.role == "tool" or message.role == "ipython" %}
        {{- "<|start_header_id|>ipython<|end_header_id|>\n\n" }}
        {%- if message.content is mapping or message.content is iterable %}
            {{- message.content | tojson }}
        {%- else %}
            {{- message.content }}
        {%- endif %}
        {{- "<|eot_id|>" }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' }}
{%- endif %}
""".strip()
tokenizer.chat_template = chat_template

## Автоматическая сборка промпта

Здесь prompt собирается автоматически через `tokenizer.apply_chat_template`.


In [6]:
messages = [
    {"role": "system", "content": "Some system message"},
    {"role": "user", "content": "This is a message from the user"},
    {"role": "assistant", "content": "this is a mesage from the assistant"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False)

In [7]:
reference_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Some system message<|eot_id|><|start_header_id|>user<|end_header_id|>

This is a message from the user<|eot_id|><|start_header_id|>assistant<|end_header_id|>

this is a mesage from the assistant<|eot_id|>"""

assert prompt == reference_prompt

Здесь проверяется, как chat template добавляет служебные поля и generation prompt для ответа ассистента.


## Походы в API

Здесь выполняются запросы к Together API напрямую через HTTP.


In [8]:
import getpass
import os

# Ключ Together берётся из переменной окружения или вводится вручную.
API_KEY = os.getenv("TOGETHER_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass.getpass("Введите TOGETHER_API_KEY: ").strip()

TOGETHER_TIMEOUT_SEC = 60


Здесь запрос отправляется через `requests` в chat completions API.


In [9]:
headers = {
    'Authorization': 'Bearer ' + API_KEY,
    'Content-Type': 'application/json',
}
url = "https://api.together.ai/v1/chat/completions"
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the capital of Britain?"}
]

data = {
    "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    "messages": messages
}

response = requests.post(url, headers=headers, json=data, timeout=TOGETHER_TIMEOUT_SEC)
response.raise_for_status()
model_answer = response.json()["choices"][0]["message"]["content"]
assert "london" in model_answer.lower()

Здесь сравниваются два режима обращения к модели: через chat messages и через сырой prompt.


In [10]:
headers = {
    'Authorization': 'Bearer ' + API_KEY,
    'Content-Type': 'application/json',
}
# Для отправки сырого промпта используем endpoint v1/completions (не v1/chat/completions)
url = "https://api.together.ai/v1/completions"
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the capital of Britain?"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

data = {
    "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    "prompt": prompt
}

response = requests.post(url, headers=headers, json=data, timeout=TOGETHER_TIMEOUT_SEC)
response.raise_for_status()
model_answer = response.json()["choices"][0]["text"]
assert "london" in model_answer.lower() and "assistant" not in model_answer.lower()

## Клиент

Здесь используется Python-клиент Together и параметры сэмплинга для генерации ответа.


In [11]:
import time

client = Together(api_key=API_KEY)


def api_call_with_retry(fn, *args, retries: int = 3, delay: float = 2.0, **kwargs):
    """
    Вызывает fn(*args, **kwargs) с timeout и повторными попытками при 503 / 529.
    Together.ai иногда возвращает Service Unavailable под нагрузкой —
    достаточно подождать пару секунд и повторить.
    """
    kwargs.setdefault("timeout", TOGETHER_TIMEOUT_SEC)
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt < retries - 1 and ("503" in str(e) or "529" in str(e) or "service_unavailable" in str(e).lower()):
                print(f"[retry {attempt + 1}/{retries}] {e} — повтор через {delay}s...")
                time.sleep(delay)
            else:
                raise

In [12]:
model_name = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the capital of Britain?"}
]

response = api_call_with_retry(
    client.chat.completions.create,
    model=model_name,
    messages=messages,
    temperature=0.5,
    top_k=100,
    top_p=0.9,
    repetition_penalty=1.05,
)

response_text = response.choices[0].message.content
assert "london" in response_text.lower()

Здесь сырой prompt отправляется через `client.completions.create`.


## Tools

Здесь описывается простая функция-инструмент, которая имитирует получение данных пользователя из базы.


In [13]:
def get_user_info_from_db(person_name: str) -> Dict[str, str]:
    database = {
        "ilya": {
            "job": "Software Developer",
            "pets": "dog",
        },
        "farruh": {
            "job": "Senior Data & Solution Architect",
            "hobby": "travelling, hiking",
        },
        "timur": {
            "job": "DeepSchool Founder",
            "city": "Novosibirsk",
        }
    }
    no_info = {"err": f"No info about {person_name}"}
    return database.get(person_name.lower(), no_info)


print(get_user_info_from_db("Timur"))

{'job': 'DeepSchool Founder', 'city': 'Novosibirsk'}


Здесь функция описывается в JSON-формате tool schema.


In [14]:
get_user_info_from_db_tool = {
    "type": "function",
    "function": {
        "name": "get_user_info_from_db",
        "description": "This function returns a json of information about the user",
        "parameters": {
            "type": "object",
            "properties": {
                "person_name": {
                    "type": "string",
                    "description": "Name of the person to look for."
                }
            },
            "required": ["person_name"]
        }
    }
}


Здесь список tools передаётся в chat template вместе с историей сообщений.


In [15]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya?"}
]
prompt = tokenizer.apply_chat_template(messages,
                                       tokenize=False,
                                       tools=[get_user_info_from_db_tool],
                                       add_generation_prompt=True)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}.Do not use variables.

{
    "type": "function",
    "function": {
        "name": "get_user_info_from_db",
        "description": "This function returns a json of information about the user",
        "parameters": {
            "type": "object",
            "properties": {
                "person_name": {
                    "type": "string",
                    "description": "Name of the person to look for."
                }
            },
            "required": [
                "person_name"
            ]
        }
    

Здесь сравнивается вызов tool-формата на маленькой и большой модели.


In [ ]:

model_small = "Qwen/Qwen2.5-7B-Instruct-Turbo"
model_large = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

In [17]:
generation_kwargs = {
    "max_tokens": 120,
    "stop": ["<|eot_id|>"],
}

response_small = api_call_with_retry(client.completions.create, model=model_small, prompt=prompt, **generation_kwargs)
response_large = api_call_with_retry(client.completions.create, model=model_large, prompt=prompt, **generation_kwargs)

print("=== model_small (Qwen 7B) ===")
print(response_small.choices[0].text)
print("\n=== model_large (Llama 70B) ===")
print(response_large.choices[0].text)

=== model_small (Qwen 7B) ===
{"name": "get_user_info_from_db", "parameters": {"person_name": "Ilya"}}

=== model_large (Llama 70B) ===
{"type": "function", "name": "get_user_info_from_db", "parameters": {"person_name": "Ilya"}}


Здесь сырой структурированный ответ модели разбирается и превращается в реальный вызов Python-функции.


In [ ]:
FUNCTION_REGISTRY = {"get_user_info_from_db": get_user_info_from_db}
reference_answer = """{"name": "get_user_info_from_db", "parameters": {"person_name": "Ilya"}}"""


def parse_function_call(model_answer):
    try:
        call = json.loads(model_answer)
    except (TypeError, json.JSONDecodeError):
        return None

    if not isinstance(call, dict):
        return None

    function_name = call.get("name")
    parameters = call.get("parameters")
    if function_name not in FUNCTION_REGISTRY or not isinstance(parameters, dict):
        return None

    try:
        return FUNCTION_REGISTRY[function_name](**parameters)
    except TypeError:
        return None


assert parse_function_call(reference_answer) == get_user_info_from_db("Ilya")


Здесь результат tool call добавляется обратно в историю диалога, после чего модель формирует финальный ответ.


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya?"}
]
model_answer = reference_answer
tool_result = parse_function_call(model_answer)
assert tool_result is not None

messages.append({"role": "assistant", "content": model_answer})
messages.append({"role": "tool", "content": tool_result})

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                       tools=[get_user_info_from_db_tool])

response = api_call_with_retry(client.completions.create, model=model_large, prompt=prompt, temperature=0.8)
print(response.choices[0].text)


This is the information I found about Ilya.


Здесь function calling проверяется через chat API и поле `tools`.


In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya?"}
]
response = api_call_with_retry(
    client.chat.completions.create,
    model=model_large,
    messages=messages,
    tools=[get_user_info_from_db_tool],
)

Здесь из ответа chat API извлекается поле `tool_calls`.


In [21]:
response.choices[0].message.tool_calls

[ToolChoice(id='call_7a2k0m7o7zadj53r6t75luei', function=Function(arguments='{"person_name":"Ilya"}', name='get_user_info_from_db'), index=0.0, type='function')]

In [22]:
import os
from langchain_together import ChatTogether
from langchain_core.messages import AIMessage
from langchain_core.tools import tool

Здесь подключается LangChain-интеграция для Together AI.


In [23]:
os.environ["TOGETHER_API_KEY"] = API_KEY

llm = ChatTogether(
    model=model_large,
    temperature=0,
    max_tokens=100,
    timeout=TOGETHER_TIMEOUT_SEC,
    max_retries=2
)

In [24]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What do you know about Ilya? Answer in one short sentence."}
]
response = llm.invoke(messages)
print(response.content)

Ilya can refer to various individuals, including Ilya Repin, a Russian painter, or Ilya Kovalchuk, a Russian professional ice hockey player.


Здесь определяется простая функция-инструмент для демонстрации tool calling.


In [25]:
def magic_operation(a, b):
    return b[::-1] + a


assert magic_operation("456", "321") == "123456"

Здесь функция оборачивается в LangChain tool с типами и docstring.


In [26]:
@tool
def magic_operation_tool(a: str, b: str) -> str:
    """
    This function takes two strings, reverses the second one and concatenates to it the first string.
    It returns the concatenation as a result.
    """
    return magic_operation(a, b)


print(magic_operation_tool.args_schema.model_json_schema())

{'description': 'This function takes two strings, reverses the second one and concatenates to it the first string.\nIt returns the concatenation as a result.', 'properties': {'a': {'title': 'A', 'type': 'string'}, 'b': {'title': 'B', 'type': 'string'}}, 'required': ['a', 'b'], 'title': 'magic_operation_tool', 'type': 'object'}


Здесь инструмент привязывается к LLM через `bind_tools`.


In [27]:
llm_with_tools = llm.bind_tools([magic_operation_tool], tool_choice="magic_operation_tool")

Здесь выполняется полный цикл tool calling: запрос к модели, вызов инструмента и финальный ответ.


In [28]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user",
     "content": "Can you help me? Do not reveal the workings of magic operation, but give me the result of it for strings `456` and `321`"}
]
resp = llm_with_tools.invoke(messages)

# LLM/tool-calling через внешний API может быть недетерминированным.
# Если модель вдруг не вернула tool call, используем явный fallback,
# чтобы дальше проверять механику вызова инструмента, а не случайность модели.
if not resp.tool_calls:
    resp = AIMessage(
        content="",
        tool_calls=[{
            "name": "magic_operation_tool",
            "args": {"a": "456", "b": "321"},
            "id": "manual_magic_operation",
        }],
    )

messages.append(resp)
messages.append(magic_operation_tool.invoke(resp.tool_calls[0]))
messages[-1]

ToolMessage(content='123456', name='magic_operation_tool', tool_call_id='call_lautsn530l4atj1dvo741sip')

In [29]:
assert len(messages) == 4

In [30]:
res = llm.invoke(messages).content
print(res)

# Проверяем детерминированный результат инструмента, а не произвольную формулировку LLM.
assert "123456" in messages[-1].content

The result of the magic operation for strings `456` and `321` is `123456`.



Здесь тот же сценарий tool calling собирается через LlamaIndex и `ReActAgent`.


In [31]:
from llama_index.llms.together import TogetherLLM
from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent

In [32]:
llm = TogetherLLM(model=model_large, api_key=API_KEY, timeout=TOGETHER_TIMEOUT_SEC)

Здесь определяется версия функции-инструмента для LlamaIndex.


In [33]:
def magic_operation_tool(a: str, b: str) -> str:
    """
    This function takes two strings, reverses the second one and concatenates to it the first string.
    It returns the concatenation as a result.
    """
    print("INSIDE FUNCTION CALL")
    return magic_operation(a, b)

Здесь функция превращается в LlamaIndex `FunctionTool`.


In [34]:
magic_operation_tool_llamaindex = FunctionTool.from_defaults(fn=magic_operation_tool)
print(magic_operation_tool_llamaindex.metadata)

ToolMetadata(description='magic_operation_tool(a: str, b: str) -> str\n\n    This function takes two strings, reverses the second one and concatenates to it the first string.\n    It returns the concatenation as a result.', name='magic_operation_tool', fn_schema=<class 'llama_index.core.tools.utils.magic_operation_tool'>, return_direct=False)


Здесь создаётся `ReActAgent` с подключённым инструментом.


In [35]:
agent = ReActAgent(tools=[magic_operation_tool_llamaindex], llm=llm, verbose=True)

In [36]:
text = "Can you help me? Do not reveal the workings of magic operation, but give me the result of it for strings `456` and `321`"
response = await agent.run(text)
print(response)

[tick] add: AgentWorkflowStartEvent(user_msg='Can you help me? Do not reveal the workings of mag...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Can you help me? Do not reveal the workings of magic operation, but g...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Can you help me? Do not reveal the workings of magic operation, but g...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_

Здесь собирается простой финансовый агент: он ищет день с максимальным изменением цены акции и сопоставляет его с новостями.


In [ ]:
import yfinance as yf
from datetime import datetime, timedelta

stock = yf.Ticker("AAPL")
YFINANCE_TIMEOUT_SEC = 15
df = stock.history(period="1mo", timeout=YFINANCE_TIMEOUT_SEC)

df["diff"] = df["Close"] - df["Open"]
df[["Open", "Close", "diff"]].round(2)

,Open,Close,diff
Date,,,
2026-04-20 00:00:00-04:00,270.08,272.80,2.72
2026-04-21 00:00:00-04:00,271.25,265.93,-5.33
2026-04-22 00:00:00-04:00,267.57,272.92,5.35
2026-04-23 00:00:00-04:00,274.80,273.18,-1.62
2026-04-24 00:00:00-04:00,272.51,270.81,-1.70
2026-04-27 00:00:00-04:00,265.85,267.36,1.52
2026-04-28 00:00:00-04:00,272.09,270.46,-1.63
2026-04-29 00:00:00-04:00,267.30,269.92,2.62
2026-04-30 00:00:00-04:00,270.25,271.10,0.85


Здесь подключается NewsAPI для поиска новостей по компании и дате.


In [ ]:
NEWS_API_KEY = os.getenv("NEWS_API_KEY", "").strip() or "API KEY"
NEWS_API_URL = "https://newsapi.org/v2/everything"
NEWS_API_TIMEOUT_SEC = 15
NEWS_API_PAGE_SIZE = 25 
NEWS_API_PLACEHOLDERS = {"", "API KEY", "..."}
YFINANCE_TIMEOUT_SEC = 15

date_to = datetime.now().strftime("%Y-%m-%d")
date_from = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")

params = {
    "q": "Apple",
    "apiKey": NEWS_API_KEY,
    "from": date_from,
    "to": date_to,
    "sortBy": "relevancy",
    "pageSize": NEWS_API_PAGE_SIZE,
    "searchIn": "title,description",
    "language": "en",
}

if NEWS_API_KEY in NEWS_API_PLACEHOLDERS:
    print("Укажите NEWS_API_KEY: задайте переменную окружения или вставьте ключ в ячейку.")
    articles = {"articles": []}
else:
    try:
        api_response = requests.get(NEWS_API_URL, params=params, timeout=NEWS_API_TIMEOUT_SEC)
        api_response.raise_for_status()
        articles = api_response.json()
    except requests.Timeout:
        print(f"NewsAPI не ответил за {NEWS_API_TIMEOUT_SEC} секунд. Попробуйте уменьшить период или повторить позже.")
        articles = {"articles": []}
    except requests.RequestException as exc:
        print(f"Ошибка запроса к NewsAPI: {exc}")
        articles = {"articles": []}
    except ValueError:
        print("NewsAPI вернул ответ не в формате JSON. Попробуйте повторить запрос позже.")
        articles = {"articles": []}

if articles.get("status") == "error":
    print(f"NewsAPI error: {articles.get('code')} — {articles.get('message')}")
else:
    for article in articles.get("articles", []):
        title = article.get("title")
        if title != "[Removed]" and article.get("description"):
            print(title)
            print(article["description"])
            break


Укажите NEWS_API_KEY: задайте переменную окружения или вставьте ключ в ячейку.


Здесь фильтруются удалённые статьи, а из оставшихся берутся заголовок и описание.


Здесь реализуется ReAct-агент с двумя инструментами: поиск максимального изменения цены акции и поиск релевантных новостей.


In [ ]:
NEWS_API_URL = "https://newsapi.org/v2/everything"
NEWS_API_TIMEOUT_SEC = 15
NEWS_API_PAGE_SIZE = 25
NEWS_API_PLACEHOLDERS = {"", "API KEY", "..."}


def get_day_with_max_diff(company_stock_name: str) -> Tuple[str, float]:
    """
    Returns the date (YYYY-MM-DD) and the price difference (Close - Open) for the day
    with the biggest intraday price movement of the given stock ticker over the past month.
    """
    stock = yf.Ticker(company_stock_name)
    df = stock.history(period="1mo", timeout=YFINANCE_TIMEOUT_SEC)
    df["diff"] = df["Close"] - df["Open"]
    idxm = df["diff"].idxmax()
    row = df.loc[idxm]
    return row.name.date().strftime("%Y-%m-%d"), float(row["diff"])


def get_news(company_name: str, date: str) -> List[str]:
    """
    Returns up to 5 news articles (title + description) about company_name
    published on the given date (YYYY-MM-DD).
    If NewsAPI is slow or unavailable, returns a short diagnostic message
    so the agent can continue instead of hanging indefinitely.
    """
    params = {
        "q": company_name,
        "apiKey": NEWS_API_KEY,
        "from": date,
        "to": date,
        "sortBy": "relevancy",
        "pageSize": NEWS_API_PAGE_SIZE,
        "searchIn": "title,description",
        "language": "en",
    }

    if NEWS_API_KEY in NEWS_API_PLACEHOLDERS:
        return ["NewsAPI key is not configured. Use stock data, say that news are unavailable, and do not infer a specific cause from news."]

    try:
        api_response = requests.get(NEWS_API_URL, params=params, timeout=NEWS_API_TIMEOUT_SEC)
        api_response.raise_for_status()
        payload = api_response.json()
    except requests.Timeout:
        return [f"NewsAPI did not respond within {NEWS_API_TIMEOUT_SEC} seconds. Use stock data, say that news are unavailable, and do not infer a specific cause from news."]
    except requests.RequestException as exc:
        return [f"NewsAPI request failed: {exc}. Use stock data, say that news are unavailable, and do not infer a specific cause from news."]
    except ValueError:
        return ["NewsAPI returned a non-JSON response. Use stock data, say that news are unavailable, and do not infer a specific cause from news."]

    if payload.get("status") == "error":
        code = payload.get("code", "unknown_error")
        message = payload.get("message", "No details")
        return [f"NewsAPI returned an error ({code}): {message}"]

    articles = payload.get("articles", [])
    good = [a for a in articles if a.get("title") != "[Removed]" and a.get("description")]
    if not good:
        return ["No relevant NewsAPI articles found for this date. Say that there is not enough news evidence to explain the move."]

    sampled = sample(good, min(5, len(good)))
    return [f"{a['title']}\n{a['description']}" for a in sampled]


llm_agent = TogetherLLM(model=model_large, api_key=API_KEY, timeout=TOGETHER_TIMEOUT_SEC)

agent = ReActAgent(
    tools=[
        FunctionTool.from_defaults(fn=get_day_with_max_diff),
        FunctionTool.from_defaults(fn=get_news),
    ],
    llm=llm_agent,
    verbose=True,
    system_prompt=(
        "You are a financial analyst assistant. "
        "When asked about a company, use get_day_with_max_diff to find the day with the biggest "
        "stock price movement, then use get_news to find news from that day, "
        "and explain what likely caused the price change. "
        "If news are unavailable or the tool returns an error, say that clearly and do not invent a news-based cause."
    ),
)

response = await agent.run("Can you check when was the biggest difference in Apple (AAPL) stock price and explain what caused it?")
print(response)

[tick] add: AgentWorkflowStartEvent(user_msg='Can you check when was the biggest difference in A...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Can you check when was the biggest difference in Apple (AAPL) stock p...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a financial analyst assistant. When asked about a company...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_